# League chat toxicity: untrained vs tuned, live

Type any League of Legends chat line, and two models label it `toxic` or `nontoxic` side by side.

| Column | Model | Pinned to commit |
| --- | --- | --- |
| untrained | [Qwen/Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) | `70d244cc86ccca08cf5af4e1e306ecf908b1ad5e` |
| tuned (v2) | the same model plus the v2 LoRA adapter, [Juchen/league-toxicity-qwen3-1.7b](https://huggingface.co/Juchen/league-toxicity-qwen3-1.7b) | `80e3d1bac9505024f49cf3439926f2be7d2e809d` |

Both columns use the prompt, chat template and greedy decoding of `eval.py` in [the repo](https://github.com/joshuatagoe/league-toxicity-slm). The untrained column is the same weights with the adapter switched off.

**To run it**
1. Runtime → Change runtime type → **T4 GPU** (free).
2. Runtime → **Run all**. The first run installs three libraries and downloads about 4 GB of weights, so expect a few minutes.
3. Type a line in the box in step 3. The cell runs again each time the text changes.

It also runs without a GPU, but slowly.

## 1. Install the pinned libraries

In [ ]:
%pip install -q transformers==5.22.0 peft==0.21.2 accelerate==1.15.0

## 2. Load both models

The eval runs used bf16. A T4 has no bf16, so on a T4 the weights load in fp16; on an L4 or A100 they load in bf16, as in the eval runs. The rounding differs a little in fp16, so a borderline line can get a different answer. Step 5 checks the answers against the recorded eval runs.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE, BASE_REVISION = "Qwen/Qwen3-1.7B", "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"
TUNED, TUNED_REVISION = "Juchen/league-toxicity-qwen3-1.7b", "80e3d1bac9505024f49cf3439926f2be7d2e809d"

if torch.cuda.is_available():
    dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    device_map = {"": 0}
else:
    dtype, device_map = torch.float32, None

tokenizer = AutoTokenizer.from_pretrained(BASE, revision=BASE_REVISION)
tokenizer.padding_side = "left"
base = AutoModelForCausalLM.from_pretrained(BASE, revision=BASE_REVISION, dtype=dtype, device_map=device_map)
model = PeftModel.from_pretrained(base, TUNED, revision=TUNED_REVISION)
model.eval()
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", "| weights:", dtype)

In [ ]:
# The prompt, decoding and answer parsing from the repo (src/slm_prompt.py, src/slm.py, src/labels.py).
import re

import pandas as pd

SYSTEM = (
    "You moderate League of Legends chat. Label the player's message toxic or nontoxic, "
    "the way experienced League players would. Reply with exactly one word: toxic or nontoxic."
)
_NEGATED = re.compile(r"\b(?:non|not)\s*[-_]?\s*toxic\b")
_LABEL = re.compile(r"\b(nontoxic|toxic)\b")


def render(text):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)


@torch.no_grad()
def generate(texts, batch_size=32, max_new_tokens=6):
    out = []
    for i in range(0, len(texts), batch_size):
        prompts = [render(t) for t in texts[i : i + batch_size]]
        enc = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        out += tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1] :], skip_special_tokens=True)
    return [o.strip() for o in out]


def parse_label(raw):
    s = raw.lower().replace("`", " ").replace('"', " ").replace("'", " ")
    m = _LABEL.search(_NEGATED.sub("nontoxic", s))
    return int(m.group(1) == "toxic") if m else None


def compare(lines):
    """Label each line with the untrained model, then with the tuned one."""
    with model.disable_adapter():
        untrained = generate(lines)
    tuned = generate(lines)
    return [{"line": l, "untrained": u, "tuned": t} for l, u, t in zip(lines, untrained, tuned)]


def answer(raw, gold=None):
    label = parse_label(raw)
    text = {1: "toxic", 0: "nontoxic"}.get(label, f"no answer ({raw!r})")
    return text if gold is None else f"{text} {'✓' if label == gold else '✗'}"


def show(rows, gold=None, labelled_by=None):
    gold = gold or [None] * len(rows)
    table = pd.DataFrame({
        "line": [r["line"] for r in rows],
        "untrained": [answer(r["untrained"], g) for r, g in zip(rows, gold)],
        "tuned (v2)": [answer(r["tuned"], g) for r, g in zip(rows, gold)],
    })
    if labelled_by:
        table.insert(1, "label", [{1: "toxic", 0: "nontoxic"}[g] for g in gold])
        table.insert(2, "labelled by", labelled_by)
    table.index = range(1, len(table) + 1)
    display(table)

## 3. Type a line

In [ ]:
# @title Type a chat line { run: "auto" }
line = "wow great gank jungle, really helpful"  # @param {type:"string"}
show(compare([line.strip()]))

In [ ]:
# @title Or several lines at once (edit the list)
lines = [
    "go back to iron noob",
    "report mid for afk",
    "jg diff",
]
show(compare([l.strip() for l in lines if l.strip()]))

## 4. Lines with known labels

These are the lines on the repo's demo page. A ✓ means the answer matches the label.
- **expert annotators:** real chat from the L2DTnH test split that the untrained model gets wrong.
- **project author:** pressure lines, labelled under `docs/LABELING_RULES.md`. The first three are the ones the v2 data fixed; the last four are sarcasm and taunts that v2 still misses.

In [ ]:
EXAMPLES = [  # (line, label: 1 = toxic, who labelled it)
    ("with this noob botlane", 1, "expert annotators"),
    ("gg easy noobs", 1, "expert annotators"),
    ("oh shut up", 1, "expert annotators"),
    ("dat team sucks hard", 1, "expert annotators"),
    ("report ashe flaming", 0, "expert annotators"),
    ("report rengar pls for troll", 0, "expert annotators"),
    ("fuck sry", 0, "expert annotators"),
    ("i got caught in jarvan's ult", 0, "expert annotators"),
    ("fucking lag again", 0, "project author"),
    ("cry more top", 0, "project author"),
    ("jg diff", 1, "project author"),
    ("wow great gank jungle, really helpful", 1, "project author"),
    ("gg ez", 1, "project author"),
    ("go play some other game pls", 1, "project author"),
    ("have u considered tft", 1, "project author"),
]
show(compare([e[0] for e in EXAMPLES]), gold=[e[1] for e in EXAMPLES], labelled_by=[e[2] for e in EXAMPLES])

## 5. Check against the recorded eval runs

Runs the 40 pressure lines through both models, in the same batch as `eval.py`, and compares each answer with the one the eval run recorded (in `runs/` on GitHub). The eval runs scored the untrained model 25/40 and the tuned one 30/40.

In [ ]:
import json
import urllib.request

REPO = "https://raw.githubusercontent.com/joshuatagoe/league-toxicity-slm/9f4fa84eaf7e77a64ed1d89717f789aabbc6fb29/"


def fetch_jsonl(path):
    text = urllib.request.urlopen(REPO + path).read().decode("utf-8")
    return [json.loads(l) for l in text.splitlines() if l.strip()]


pressure = fetch_jsonl("data/pressure/PRESSURE.jsonl")
runs = {"untrained": "qwen3-1.7b-base__slm__na", "tuned": "qwen3-1.7b-v2__slm__na"}
recorded = {name: {r["id"]: r["pred"] for r in fetch_jsonl(f"runs/{run}__PRESSURE.jsonl")} for name, run in runs.items()}
rows = compare([r["message"] for r in pressure])
for name in runs:
    preds = [parse_label(r[name]) for r in rows]
    right = sum(p == r["label"] for p, r in zip(preds, pressure))
    same = sum(p == recorded[name][r["id"]] for p, r in zip(preds, pressure))
    print(f"{name}: {right}/{len(pressure)} right; same answer as the eval run on {same}/{len(pressure)}")
    for p, r, row in zip(preds, pressure, rows):
        if p != recorded[name][r["id"]]:
            was = {1: "toxic", 0: "nontoxic"}.get(recorded[name][r["id"]], "no answer")
            print(f"  differs: {r['message']!r} is {answer(row[name])} here, {was} in the eval run")

## Scoring a whole file

For the staff held-out set, or any file of labelled chat lines, use `compare.py` in the repo. It runs both models on the file and prints precision, recall, F1, the paired difference with a 95% interval, and every line where the two disagree:

```
python compare.py --eval-set path/to/heldout.csv
```